# 02 — Classical 3-Bit Repetition Code

## Read before you begin

- Arthur Pesah, [*Introduction to Quantum Error Correction, Part 1*](https://arthurpesah.me/blog/2022-01-25-intro-qec-1/).
- Exact sections to read: **1. A bit of history**, **2.1 Modeling quantum noise**, **2.2 Encoding and decoding**.
- Concepts to extract: the historical motivation for error correction (the threshold theorem); why *any* physical noise process can be "digitized" into bit-flip ($X$) and phase-flip ($Z$) errors — a forward reference to notebook 03's Pauli operators; and the 3-repetition code plus the idea of a non-destructive parity ("stabilizer") measurement, which this notebook now builds rigorously in pure classical bits, before notebook 03 makes it quantum.

## Goal

Build the simplest error-correcting code — the classical 3-bit repetition code — entirely from first principles, using nothing but plain Python bits. This notebook is the bridge between "parity checks on a Stim ancilla" (notebook 01) and the quantum stabilizer formalism (notebook 03).

By the end of this notebook you will be able to:

1. define encoding, code distance, and the difference between detecting and correcting errors;
2. implement `encode`, `flip_bit`, `syndrome`, and `decode` for the 3-bit repetition code;
3. read a generated table showing how each single-bit-flip error is detected, located, and corrected;
4. explain why a *2*-bit repetition code can only detect — not correct — a single error.

**Expected outputs:** a syndrome/decoding table for the four single-error cases on a 3-bit codeword, and a short demonstration of the 2-bit code's ambiguity.

**How to use this notebook:** run cells in order. Before a cell marked **Prediction**, stop and write down your own answer before running the next cell.

## 1. Definitions

- **Physical qubit** — a single physical two-level quantum system used to store one qubit of information. *This notebook deliberately uses plain classical bits instead of qubits* — the coding-theory ideas (redundancy, syndromes, majority vote) are identical, and it is easier to see them without also tracking superposition and measurement collapse. Notebook 03 will translate everything here into physical qubits and Pauli operators.
- **Logical qubit** (here: **logical bit**) — the single unit of information you actually care about protecting. It is *not* stored in one physical bit; it is encoded redundantly across several.
- **Encoding** — the fixed rule that maps a logical value to a longer string of physical values (a *codeword*). Encoding trades physical resources (more bits) for resilience to noise.
- **Code distance** — the minimum number of single-bit flips needed to turn one valid codeword into a *different* valid codeword. Distance $d$ lets you:
  - **detect** up to $d-1$ errors (you notice something is wrong), and
  - **correct** up to $\lfloor (d-1)/2 \rfloor$ errors (you can tell exactly what went wrong and undo it).
- **Error detection vs. error correction** — detection means a check tells you *"an error occurred"* (or *"the state is no longer a valid codeword"*) without necessarily telling you where. Correction means you can pinpoint *which* physical bit(s) flipped and reverse the flip, recovering the original logical value.

## 2. The 3-bit repetition code

The encoding rule is simply "repeat the bit three times":

$$0 \rightarrow 000 \qquad 1 \rightarrow 111$$

The two valid codewords, `000` and `111`, differ in all 3 positions, so the code distance is $d = 3$.

Using the formulas above:

- detects up to $d - 1 = 2$ bit flips,
- corrects up to $\lfloor (d-1)/2 \rfloor = 1$ bit flip.

That second bound is the important one: this code guarantees a correct recovery only when **at most one** of the three bits has flipped. Section 7 revisits what happens when that assumption is violated.

In [1]:
def encode(logical_bit):
    """Encode a single logical bit (0 or 1) as a 3-bit repetition codeword."""
    if logical_bit not in (0, 1):
        raise ValueError("logical_bit must be 0 or 1")
    return (logical_bit, logical_bit, logical_bit)


for bit in (0, 1):
    print(f"encode({bit}) = {encode(bit)}")

encode(0) = (0, 0, 0)
encode(1) = (1, 1, 1)


### Prediction 1

Suppose you encode logical `1` as `111` and then flip exactly one bit.

**Before running the next cells:** what do you expect the two parity checks $b_0 \oplus b_1$ and $b_1 \oplus b_2$ to output if the flipped bit is bit 0? Bit 1? Bit 2? Will the two checks ever agree (both 0, or both 1) for a *single* flipped bit?

In [2]:
def flip_bit(codeword, index):
    """Return a copy of codeword with the bit at `index` flipped."""
    bits = list(codeword)
    bits[index] ^= 1
    return tuple(bits)


codeword = encode(1)
for i in range(3):
    print(f"flip_bit({codeword}, {i}) = {flip_bit(codeword, i)}")

flip_bit((1, 1, 1), 0) = (0, 1, 1)
flip_bit((1, 1, 1), 1) = (1, 0, 1)
flip_bit((1, 1, 1), 2) = (1, 1, 0)


## 3. Syndrome: parity checks

Two parity checks are enough to cover all 3 bits of a 3-bit codeword:

$$s_0 = b_0 \oplus b_1 \qquad s_1 = b_1 \oplus b_2$$

Each check compares a neighboring pair of bits. If the codeword is undisturbed (`000` or `111`), both checks return 0, since neighboring bits always agree in a valid codeword. A single bit flip breaks agreement with its neighbor(s), producing a nonzero syndrome $(s_0, s_1)$.

In [3]:
def syndrome(codeword):
    """Compute the 2-bit syndrome (s0, s1) for a 3-bit codeword."""
    b0, b1, b2 = codeword
    s0 = b0 ^ b1
    s1 = b1 ^ b2
    return (s0, s1)


for i in range(3):
    flipped = flip_bit(codeword, i)
    print(f"codeword={flipped}, syndrome={syndrome(flipped)}")

codeword=(0, 1, 1), syndrome=(1, 0)
codeword=(1, 0, 1), syndrome=(1, 1)
codeword=(1, 1, 0), syndrome=(0, 1)


Notice that each of the three single-bit-flip syndromes is distinct, and distinct from `(0, 0)`. That is exactly what lets us *locate* the error, not just detect it — the lookup table below turns each syndrome into an error location.

In [4]:
# Syndrome -> which bit (if any) is inferred to have flipped.
SYNDROME_TABLE = {
    (0, 0): None,  # no error
    (1, 0): 0,     # only the b0/b1 check fired -> bit 0 flipped
    (1, 1): 1,     # both checks fired -> bit 1 flipped
    (0, 1): 2,     # only the b1/b2 check fired -> bit 2 flipped
}


def locate_error(codeword):
    """Return the index of the bit inferred to have flipped, or None."""
    return SYNDROME_TABLE[syndrome(codeword)]


for i in range(3):
    flipped = flip_bit(codeword, i)
    print(f"actual flip: bit {i}, inferred: bit {locate_error(flipped)}")

actual flip: bit 0, inferred: bit 0
actual flip: bit 1, inferred: bit 1
actual flip: bit 2, inferred: bit 2


### Prediction 2

`decode` will recover the logical bit by majority vote: whichever value (0 or 1) appears in at least two of the three positions.

**Before running the next cell:** if exactly one bit has flipped, will majority vote always recover the original logical bit? What if the syndrome is `(0, 0)` but the codeword is `111` — is a logical `1` or `0`?

In [5]:
def decode(codeword):
    """Decode a (possibly corrupted) 3-bit codeword by majority vote."""
    return 1 if sum(codeword) >= 2 else 0


for i in range(3):
    flipped = flip_bit(codeword, i)
    print(f"codeword={flipped}, decoded logical bit={decode(flipped)}")

codeword=(0, 1, 1), decoded logical bit=1
codeword=(1, 0, 1), decoded logical bit=1
codeword=(1, 1, 0), decoded logical bit=1


## 4. Full syndrome / decoding table

Putting `flip_bit`, `syndrome`, `locate_error`, and `decode` together for the four cases that a distance-3 code is designed to handle: no error, and a single flip on each of the three bits.

In [6]:
logical_bit = 1
base_codeword = encode(logical_bit)

cases = {
    "no error": base_codeword,
    "flip bit 0": flip_bit(base_codeword, 0),
    "flip bit 1": flip_bit(base_codeword, 1),
    "flip bit 2": flip_bit(base_codeword, 2),
}

header = f"{'case':<12} {'received':>10} {'syndrome':>10} {'decoded bit':>13} {'inferred error':>15}"
print(header)
print("-" * len(header))
for label, received in cases.items():
    s = syndrome(received)
    decoded = decode(received)
    inferred = locate_error(received)
    inferred_str = "none" if inferred is None else f"bit {inferred}"
    received_str = "".join(str(b) for b in received)
    print(f"{label:<12} {received_str:>10} {str(s):>10} {decoded:>13} {inferred_str:>15}")

case           received   syndrome   decoded bit  inferred error
----------------------------------------------------------------
no error            111     (0, 0)             1            none
flip bit 0          011     (1, 0)             1           bit 0
flip bit 1          101     (1, 1)             1           bit 1
flip bit 2          110     (0, 1)             1           bit 2


Every single-bit-flip row correctly identifies the flipped bit and decodes back to the original logical bit (`1`). This is the code doing exactly what its distance promises: correcting any single error.

### Prediction 3

Now imagine a *2*-bit repetition code: `0 -> 00`, `1 -> 11`, with a single parity check $s = b_0 \oplus b_1$.

**Before running the next cell:** if bit 0 flips, what is $s$? If instead bit 1 flips, what is $s$? Given only the value of $s$, can you tell *which* bit flipped?

## 5. Why a 2-bit code can detect but not correct

A 2-bit repetition code has only two valid codewords, `00` and `11`, which differ in both positions — code distance $d = 2$. By the formulas in Section 1, distance 2 detects up to $d - 1 = 1$ error but corrects up to $\lfloor (d-1)/2 \rfloor = 0$ errors. The demonstration below shows *why*: flipping either bit produces the *same* syndrome, so the single check cannot distinguish the two possible error locations.

In [7]:
def encode2(logical_bit):
    """Encode a logical bit as a 2-bit repetition codeword."""
    return (logical_bit, logical_bit)


def syndrome2(codeword):
    """Single parity check for the 2-bit code."""
    b0, b1 = codeword
    return b0 ^ b1


base2 = encode2(0)  # (0, 0)
flip_a = (base2[0] ^ 1, base2[1])       # flip bit 0 -> (1, 0)
flip_b = (base2[0], base2[1] ^ 1)       # flip bit 1 -> (0, 1)

print(f"flip bit 0: codeword={flip_a}, syndrome={syndrome2(flip_a)}")
print(f"flip bit 1: codeword={flip_b}, syndrome={syndrome2(flip_b)}")
print("Same syndrome for both cases:", syndrome2(flip_a) == syndrome2(flip_b))

flip bit 0: codeword=(1, 0), syndrome=1
flip bit 1: codeword=(0, 1), syndrome=1
Same syndrome for both cases: True


Both single-bit flips give syndrome `1`: the check confirms *something* is wrong, but `flip_a` and `flip_b` are indistinguishable from the syndrome alone, so there is no rule that reliably picks the right bit to flip back. (It gets worse if *both* bits flip: `(0, 0) -> (1, 1)`, which is a valid codeword for the *other* logical value — the error is silently undetected.) This is precisely why the 3-bit code needs *two* parity checks: one check alone can never localize an error among more than two possibilities.

## 6. Checkpoint

Before moving on, answer these in your own words:

1. Why does code distance 3 guarantee correction of exactly one error but not two?
2. Why does the 3-bit code need *two* parity checks, while the 2-bit code only has one?
3. What would happen to the syndrome table in Section 4 if the two checks were `b0 XOR b2` and `b0 XOR b1` instead? Would it still be able to locate every single-bit error?

Once these feel clear, work through the exercises below before moving to notebook 03, which replaces classical bits with qubits and rewrites `syndrome` as an actual stabilizer measurement, connecting this table directly to the parity-measurement circuit from notebook 01.

## Exercises

These are for you to complete — the cells below intentionally only contain a stub. Fill them in, then run them.

**1.** Write `flip_multiple(codeword, indices)` that flips every bit at the given indices (reuse `flip_bit` in a loop, or implement directly). Then use it to flip **two** of the three bits in `encode(1)` and run it through `syndrome`, `locate_error`, and `decode`. Does `decode` still return the correct logical bit? Does `locate_error` return a sensible answer? Write a one-line comment explaining what you observe, in terms of the code's correction capability (Section 2).

In [8]:
def flip_multiple(codeword, indices):
    for index in indices:
        codeword = flip_bit(codeword, index)
    return codeword



# TODO: once flip_multiple is implemented, uncomment and inspect the result.
two_flips = flip_multiple(encode(1), (0, 1))
print(two_flips, syndrome(two_flips), locate_error(two_flips), decode(two_flips))

(0, 0, 1) (0, 1) 2 0


**2.** Write a function `logical_error_rate(trials, flip_probability)` that, for `trials` repetitions: encodes a random logical bit, independently flips each of the 3 physical bits with probability `flip_probability`, decodes the result, and counts how often the decoded bit differs from the original. Return the fraction of trials where decoding failed. You will need the `random` module from the standard library. Try a small `flip_probability` (e.g. 0.05) and a larger one (e.g. 0.4) — does the failure rate behave the way you'd expect from a code that only corrects a single flip?

In [9]:
import random


def logical_error_rate(trials, flip_probability):
    logical_errors = 0
    for trial in range(trials):
        codeword = encode(1)
        # Determine which bits to flip based on flip_probability
        indices_to_flip = [i for i in range(3) if random.random() < flip_probability]
        corrupted_codeword = flip_multiple(codeword, indices_to_flip)
        decoded_bit = decode(corrupted_codeword)
        if decoded_bit != 1:
            logical_errors += 1
    return logical_errors / trials


# TODO: once implemented, try e.g.:
print(logical_error_rate(10_000, 0.05))
print(logical_error_rate(10_000, 0.4))

0.008
0.3427
